# Fantasy points

Reads the week's predictions and calculates fantasy points and other stats. Run every week.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd

from src import config, external

YEAR, WEEK = config.CURRENT_YEAR, config.WEEK
print(f'{YEAR} week {WEEK}')

## 1. Load the week's predictions
Saved by `03_predict_week`.

In [ ]:
predictions = pd.read_csv(config.predictions_path(YEAR, WEEK))
print(f'{len(predictions)} players')

## 2. Fantasy points
Receiving points only (`config.SCORING`): 0.1 per yard, 6 per TD, plus 0.5 (half PPR) or 1 (PPR) per reception. The models don't predict rushing, so the occasional WR carry isn't counted.

In [ ]:
for scoring, points in config.SCORING.items():
    predictions[f'proj_{scoring}'] = sum(predictions[f'pred_{stat}'] * pts for stat, pts in points.items())

## 3. FantasyPros and injuries
- **FantasyPros**: this week's PPR WR expert consensus. `fantasypros_ppr` is their projected PPR points for the player's ranking (`r2p_pts`). Players they don't rank (often because they're out) are left empty.
- **Injury status** from this week's injury report. The model doesn't know about injuries, so an `Out` player still gets a projection, and the WR who replaces him doesn't get the extra targets.

In [ ]:
predictions, fp_date = external.add_fantasypros(predictions)
print(f'FantasyPros rankings from {fp_date}: {predictions["fantasypros_ppr"].notna().sum()} of {len(predictions)} players matched')

# Rankings from another week would list a different opponent
matched = predictions.dropna(subset=['fp_opponent'])
wrong_week = (matched['fp_opponent'] != matched['opponent_team']).mean()
if wrong_week > 0.1:
    print(f'WARNING: {wrong_week:.0%} of FantasyPros opponents differ from ours, the rankings may be for another week')

predictions = predictions.merge(external.load_injury_status(YEAR, WEEK), on='player_id', how='left')
predictions['ppr_diff'] = predictions['proj_ppr'] - predictions['fantasypros_ppr']
predictions[predictions['injury_status'].notna()][['player_display_name', 'ball_team', 'injury_status', 'fantasypros_ppr']]

## 4. Final table
`ppr_diff` is ours minus FantasyPros: positive means we project more than the experts.

In [ ]:
final = predictions.rename(columns={
    'player_display_name': 'player', 'ball_team': 'team', 'opponent_team': 'opponent',
    'pred_receptions': 'proj_receptions', 'pred_receiving_yards': 'proj_receiving_yards',
    'pred_receiving_tds': 'proj_receiving_tds',
})[['player', 'team', 'opponent', 'injury_status',
    'proj_receptions', 'proj_receiving_yards', 'proj_receiving_tds', 'proj_half_ppr', 'proj_ppr',
    'fantasypros_ppr', 'ppr_diff']]
final = final.sort_values('proj_ppr', ascending=False).reset_index(drop=True).round(2)

path = config.fantasy_path(YEAR, WEEK)
final.to_csv(path, index=False)
print('Saved', path)
final.head(30)

## 5. Biggest disagreements with FantasyPros

In [ ]:
disagree = final.dropna(subset=['ppr_diff'])
print('We project more:')
display(disagree.nlargest(10, 'ppr_diff'))
print('FantasyPros projects more:')
display(disagree.nsmallest(10, 'ppr_diff'))